In [65]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from matplotlib import pyplot as plt
import os
from tqdm.autonotebook import tqdm

C:\Users\Administrator\AppData\Local\Temp\ipykernel_40484\3405216990.py:7: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [94]:
country_raster_1km_ = rio.open(r'..\results\country-rasters\country-raster-eck4-1km.tif') 
country_raster_10km_ = rio.open(r'..\results\country-rasters\country-raster-eck4-10km.tif')

In [ ]:
# taxon = 'birds'

In [ ]:
def upscale_AOH(taxon):

    catalogue = pd.read_csv(f'../results/area-of-habitat/catalogue-{taxon}.csv')
    valid_species = catalogue[catalogue['have_a_AOH'] == True]
    catalogue_10km = valid_species.copy()
    catalogue_10km['row_min'] = np.nan
    catalogue_10km['row_max'] = np.nan
    catalogue_10km['col_min'] = np.nan
    catalogue_10km['col_max'] = np.nan

    for i in tqdm(valid_species.index):
        row_min_1km = catalogue.loc[i, 'row_min']
        row_max_1km = catalogue.loc[i, 'row_max']
        col_min_1km = catalogue.loc[i, 'col_min']
        col_max_1km = catalogue.loc[i, 'col_max']

        row_min_10km = (row_min_1km + 9) // 10
        row_max_10km = (row_max_1km + 9) // 10
        col_min_10km = (col_min_1km + 8) // 10
        col_max_10km = (col_max_1km + 8) // 10

        # print(row_min_1km, row_max_1km, col_min_1km, col_max_1km)
        # print(row_min_10km, row_max_10km, col_min_10km, col_max_10km)

        # print(949*10, 9481 + 9, 950*10, 963*10, 9623+9, 964*10)
        # print(2756*10, 27558 + 8, 2757*10, 2771*10, 27705+8, 2772*10)

        offset_x = col_min_1km + 8 - col_min_10km * 10
        offset_y = row_min_1km + 9 - row_min_10km * 10
        # print(offset_x, offset_y)

        a_species = catalogue.loc[i, 'name']
        a_src = rio.open(f'../results/area-of-habitat/{taxon}/{a_species}.tif')
        a_AOH_1km = a_src.read(1)

        H_1km = row_max_1km - row_min_1km + 1
        W_1km = col_max_1km - col_min_1km + 1
        H_10km = row_max_10km - row_min_10km + 1
        W_10km = col_max_10km - col_min_10km + 1
        # a_AOH_10km = np.zeros(shape=(H_10km, W_10km))
        a_AOH_10km_to_1km = np.zeros(shape=(H_10km * 10, W_10km * 10))
        # print(a_AOH_10km_to_1km.shape)
        a_AOH_10km_to_1km[offset_y:offset_y+H_1km, offset_x:offset_x+W_1km] = a_AOH_1km
        a_AOH_10km = a_AOH_10km_to_1km.reshape(H_10km, 10, W_10km, 10).max(axis=(1, 3)).astype(np.uint8)

        transform_10km = country_raster_10km_.transform

        x0 = transform_10km.c + col_min_10km * transform_10km.a
        y0 = transform_10km.f + row_min_10km * transform_10km.e

        new_transform = rio.Affine(transform_10km.a, 0, x0,
                                0, transform_10km.e, y0)

        output_path = os.path.join('..', 'results', 'area-of-habitat-10km', taxon, f'{a_species}.tif')
        with rio.open(
                output_path,
                'w',
                driver='GTiff',
                height=a_AOH_10km.shape[0],
                width=a_AOH_10km.shape[1],
                count=1,
                dtype=a_AOH_10km.dtype, 
                crs=country_raster_10km_.crs,
                transform=new_transform,
                compress='lzw',
            ) as dst:
                dst.write(a_AOH_10km, 1)

        catalogue_10km.loc[i, 'row_min'] = int(row_min_10km)
        catalogue_10km.loc[i, 'row_max'] = int(row_max_10km)
        catalogue_10km.loc[i, 'col_min'] = int(col_min_10km)
        catalogue_10km.loc[i, 'col_max'] = int(col_max_10km)
        catalogue_10km.to_csv(os.path.join('..', 'results', 'area-of-habitat-10km', f'catalogue-{taxon}.csv'), index=False)

In [100]:
upscale_AOH('mammals')

  0%|          | 0/5237 [00:00<?, ?it/s]

In [104]:
upscale_AOH('birds')

  0%|          | 0/9927 [00:00<?, ?it/s]

In [101]:
upscale_AOH('reptiles')

  0%|          | 0/7948 [00:00<?, ?it/s]

In [103]:
upscale_AOH('amphibians')

  0%|          | 0/6215 [00:00<?, ?it/s]